# STPPY Quick Start: the paper, end to end

Reproduces *Uncovering the spatiotemporal structure of neural avalanches through optimal transport and
dynamic time warping* from the preprocessed PsiConnect recordings:

| step | section | paper |
|---|---|---|
| 1-2 | load, preprocess, detect avalanches | Section 2.1 numbers |
| 3 | UOT + subDTW distances | (computed or loaded) |
| 4 | clustering, consensus templates, pole arrows | Figure 1, Table 1 counts, travelling/standing |
| 5 | back-fitting, sequences | input of Section 2.2 |
| 6 | paper scripts | Figures 2-4, 8-10, Tables 1-4, all statistics |

All locations are variables in `paper_scripts/paper_config.py` (`RAW_DIR`, `RUN_DIR`, `DISTANCE_DIR`,
`RESULTS_DIR`, `SURROGATE_DIR`, `TEST_RETEST_DIR`); edit them there or set the environment variables
of the same name. Run the notebook from the repository root. Figures 5-7 are schematic illustrations.

**Environment.** `environment.yml` (conda, linux-64) reproduces the paper bit for bit: Python 3.11, numpy
2.0.1 built against MKL, scipy 1.16.0, mne 1.10.1. Other builds change the preprocessed signals in the
last bits and can move single threshold crossings (e.g. 5107 instead of 5108 avalanches).

**Runtime.** Steps 1-2 ~10 min; step 3 ~370 CPU-hours (split over `N_WORKERS`) unless the distances
are found in `DISTANCE_DIR`; step 5 ~4.5 CPU-hours; step 6 ~30 min. Steps 1-5 need ~100 GB of RAM.

In [ ]:
import contextlib
import gc
import io
import multiprocessing as mp
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = Path.cwd()                                   # run from the repository root
sys.path[:0] = [str(REPO), str(REPO / 'paper_scripts')]

import mne
import numpy as np
import scipy
import stppy
import paper_config as cfg

assert Path(stppy.__file__).resolve().is_relative_to(REPO), f'stppy imported from {stppy.__file__}'
mne.set_log_level('ERROR')
print(f'stppy {stppy.__version__} from {Path(stppy.__file__).parent}')
print(f'python {sys.version.split()[0]}, numpy {np.__version__}, scipy {scipy.__version__}, mne {mne.__version__}')

RUN_DIR, RESULTS_DIR = cfg.RUN_DIR, cfg.RESULTS_DIR
RUN_DIR.mkdir(parents=True, exist_ok=True)
POS_FILE = str(RUN_DIR / 'electrode_positions.json')
N_WORKERS = int(os.environ.get('N_WORKERS', 4))     # processes for distances and back-fitting
# Distances are loaded from DISTANCE_DIR when present, otherwise computed and saved there
COMPUTE_DISTANCES = not all((cfg.DISTANCE_DIR / f).exists()
                            for f in ('peak_avalanches.pkl', 'similarity_matrix.pkl', 'path_matrix.pkl'))
MAKE_EXTRA_FIGURES = False                          # per-cluster topomaps and videos (not in the paper)
SFREQ = 250.0
for name in ('RAW_DIR', 'RUN_DIR', 'DISTANCE_DIR', 'RESULTS_DIR', 'SURROGATE_DIR', 'TEST_RETEST_DIR'):
    print(f'{name:16s} {getattr(cfg, name)}')
print(f'distances will be {"computed" if COMPUTE_DISTANCES else "loaded"}')


@contextlib.contextmanager
def quiet():
    """Silence the per-recording / per-cluster progress prints of the pipeline functions."""
    with contextlib.redirect_stdout(io.StringIO()):
        yield


def save(obj, name):
    stppy.io.save_pickle(obj, str(RUN_DIR / name))

## 1. Load and preprocess the EEG

In [ ]:
files = stppy.io.find_eeg_files(str(cfg.RAW_DIR))
dataset_names = [os.path.basename(f) for f in files]

# Electrode positions from a recording with all electrodes present
raw_complete = stppy.io.load_eeg_raw(files[7], preload=True)
stppy.io.create_electrode_positions_file(raw_complete, POS_FILE, overwrite=True)

datasets = []
for file_path in files:
    raw = stppy.io.load_eeg_raw(file_path, preload=True)
    raw_interp, _ = stppy.preprocessing.interpolate_missing_electrodes(
        raw, positions_file=POS_FILE,
        apply_average_reference=False,  # OFF: 5108 avalanches (ON gives 5333)
        verbose=False)
    datasets.append(stppy.preprocessing.resample_data(raw_interp, 250).get_data())

save(datasets, 'datasets.pkl')
save(dataset_names, 'files.pkl')
print(f'{len(datasets)} recordings, {datasets[0].shape[0]} channels at {SFREQ:.0f} Hz')

## 2. Detect avalanches

In [ ]:
crossover_thresholds = [-3, 3]
filter_masks = stppy.detection.create_avalanche_filter(datasets, crossover_thresholds)
all_avalanches, all_avalanche_times = stppy.detection.detect_avalanches(
    datasets, filter_masks, crossover_thresholds, min_duration=11, verbose=False)

# Spatial filter (5 of 8 nearest neighbours, both polarities) and sign encoding
K, channel_names = stppy.utils.create_eeg_ground_metric(POS_FILE)
with quiet():
    filtered_avalanches, filtered_times = stppy.detection.filter_avalanche_propagation(
        all_avalanches, all_avalanche_times, K, POS_FILE, min_active_neighbors=5)
peak_avalanches = stppy.detection.convert_to_peaks(filtered_avalanches)
save(peak_avalanches, 'peak_avalanches.pkl')
save(filtered_times, 'peak_avalanche_times.pkl')

# Section 2.1
n_avalanches = sum(len(r) for r in peak_avalanches)
covered = sum(end - start for rec in filtered_times for start, end in rec) / SFREQ
total = sum(d.shape[1] for d in datasets) / SFREQ
print(f'{n_avalanches} avalanches covering {covered:.2f} s = {100 * covered / total:.2f} % '
      f'of {total / 3600:.1f} h of recordings')

## 3. Distances between avalanches (UOT + subDTW)

Every avalanche is paired with its polarity-reversed copy (10216 arrays). The full matrix costs
~370 CPU-hours: it is computed when `DISTANCE_DIR` holds no distances (and saved there), otherwise
loaded after checking that it was computed for exactly these avalanches.

In [ ]:
with quiet():
    avalanches, avalanche_times = stppy.io.load_avalanche_data(
        str(RUN_DIR / 'peak_avalanches.pkl'), min_time_length=11, include_flipped=True,
        times_filepath=str(RUN_DIR / 'peak_avalanche_times.pkl'))
K_norm = stppy.utils.normalize_distance_matrix(K, method='mean')
print(f'{len(avalanches)} avalanches including polarity-reversed copies')

_BLOCK = {}


def _init_blocks(av, kn, plan):
    _BLOCK.update(av=av, kn=kn, plan=plan)


def _block(i):
    rs, re, cs, ce, diag = _BLOCK['plan'][i]
    with quiet():
        return stppy.alignment.compute_similarity_block(
            _BLOCK['av'], _BLOCK['kn'], row_start=rs, row_end=re, col_start=cs, col_end=ce,
            is_diagonal=diag)


if COMPUTE_DISTANCES:
    with quiet():
        plan = stppy.alignment.generate_block_plan(len(avalanches), block_size=int(np.ceil(len(avalanches) / 27)))
    with mp.get_context('fork').Pool(N_WORKERS, _init_blocks, (avalanches, K_norm, plan)) as pool:
        blocks = list(pool.imap_unordered(_block, range(len(plan))))
    with quiet():
        similarity_matrix, path_matrix = stppy.alignment.reconstruct_full_matrices(
            {'blocks': blocks, 'n_blocks': len(blocks)}, n_avalanches=len(avalanches))
    cfg.DISTANCE_DIR.mkdir(parents=True, exist_ok=True)
    for obj, name in ((similarity_matrix, 'similarity_matrix.pkl'), (path_matrix, 'path_matrix.pkl'),
                      (peak_avalanches, 'peak_avalanches.pkl')):
        stppy.io.save_pickle(obj, str(cfg.DISTANCE_DIR / name))
    del blocks
else:
    stored = stppy.io.load_pickle(str(cfg.DISTANCE_DIR / 'peak_avalanches.pkl'))
    same = len(stored) == len(peak_avalanches) and all(
        len(a) == len(b) and all(np.array_equal(x, y) for x, y in zip(a, b))
        for a, b in zip(stored, peak_avalanches))
    if not same:
        raise RuntimeError('the distances in DISTANCE_DIR belong to other avalanches: '
                           'point DISTANCE_DIR to an empty folder to recompute them')
    similarity_matrix = stppy.io.load_pickle(str(cfg.DISTANCE_DIR / 'similarity_matrix.pkl'))
    path_matrix = stppy.io.load_pickle(str(cfg.DISTANCE_DIR / 'path_matrix.pkl'))
save(similarity_matrix, 'similarity_matrix.pkl')
print(f'distance matrix {similarity_matrix.shape}')

## 4. Clusters, consensus templates and pole arrows (Figure 1)

In [ ]:
with quiet():
    clusters, cluster_info = stppy.clustering.simple_hierarchical_clustering(
        similarity_matrix, threshold=3.5, method='average', min_cluster_size=30)
    cluster_averages = {}
    for cluster_id in range(len(cluster_info)):
        _, average, _, _, _ = stppy.clustering.build_consensus_alignment(
            cluster_id=cluster_id, clusters=clusters, avalanche_arrays=avalanches,
            avalanche_paths=path_matrix, similarity_clean=similarity_matrix)
        cluster_averages[cluster_id] = average
    # Each avalanche has a polarity-reversed copy, so clusters come in mirror pairs: keep one of each
    clusters_to_keep = stppy.clustering.remove_duplicate_polarities(cluster_averages, threshold=-0.999)
del path_matrix
gc.collect()

kept = {cid: cluster_averages[cid] for cid in clusters_to_keep}
sizes = {cid: int((clusters == cid).sum()) for cid in clusters_to_keep}
tightness = {d['cluster_id']: d['tightness'] for d in cluster_info}
save(clusters, 'cluster_assignments.pkl')
save(kept, 'cluster_averages.pkl')
save(avalanches, 'avalanches.pkl')
save(filtered_times, 'avalanche_times.pkl')

n_clustered = sum(sizes.values())
print(f'{len(cluster_info)} clusters at threshold 3.5, {len(kept)} after removing polarity mirrors')
print(f'{n_clustered} of {n_avalanches} avalanches ({100 * n_clustered / n_avalanches:.1f} %) in the {len(kept)} clusters')
for cid in clusters_to_keep:
    print(f'  cluster {cid:2d}: n = {sizes[cid]:3d}, tightness {tightness[cid]:.2f}, '
          f'{kept[cid].shape[1]} frames')

In [ ]:
from stppy.jacobian import pole_arrows
from stppy.visualization.cluster_arrows import topomap_coords

# Travelling vs standing (Methods 4.6): stronger pole's net arrow relative to the longest one
_, xy = topomap_coords(POS_FILE, next(iter(kept.values())).shape[0])
strength = {cid: max(a.magnitude for a in pole_arrows(kept[cid], xy).values()) for cid in kept}
longest = max(strength.values())
for cid in kept:
    label = 'travelling' if strength[cid] >= 0.7 * longest else 'standing'
    print(f'  cluster {cid:2d}: {strength[cid] / longest:.2f} of the longest arrow -> {label}')

# Figure 1
fig = stppy.visualization.plot_clusters_with_arrows(
    kept, POS_FILE, cluster_sizes=sizes, sfreq=SFREQ, title='Discovered Clusters', classify=False)
out = cfg.results('figure1')
fig.savefig(out / 'figure1_discovered_clusters.png', dpi=600, bbox_inches='tight')
fig.savefig(out / 'figure1_discovered_clusters.pdf', bbox_inches='tight')

## 5. Back-fitting: sequences of templates in the continuous EEG

Every template slides over every recording (frame-wise spatial correlation); windows with
|r| > 0.825 are chained into sequences when the next one starts 0-1 frame after the previous window
ends (integer samples, stppy 0.2.0).

In [ ]:
_SCAN = {}


def _init_scan(data, templates):
    _SCAN.update(data=data, templates=templates)


def _scan(task):
    cid, ds = task
    res = stppy.detection.detect_sequences_by_correlation(_SCAN['data'][ds], _SCAN['templates'][cid], sfreq=SFREQ)
    keep = np.abs(res['correlations']) > 0.8          # superset of the 0.825 used below
    return cid, ds, res['times'][keep], res['polarities'][keep], res['correlations'][keep]


tasks = [(cid, ds) for cid in clusters_to_keep for ds in range(len(datasets))]
t0 = time.time()
with mp.get_context('fork').Pool(N_WORKERS, _init_scan, (datasets, kept)) as pool:
    scanned = pool.map(_scan, tasks, chunksize=4)
cluster_activation_times = {cid: [] for cid in clusters_to_keep}
for cid, ds, t, p, c in sorted(scanned, key=lambda r: (r[0], r[1])):
    cluster_activation_times[cid].append((ds, t, p, c))
cluster_durations = {cid: kept[cid].shape[1] / SFREQ for cid in clusters_to_keep}

with quiet():
    sequences_by_cluster = stppy.detection.detect_sequences_in_clusters(
        cluster_activation_times, cluster_durations, dataset_names=dataset_names,
        tolerance_samples=1, sfreq=SFREQ, min_sequence_length=2, correlation_threshold=0.825)
save(sequences_by_cluster, 'sequences_by_cluster.pkl')
print(f'{sum(len(v) for v in sequences_by_cluster.values())} sequences '
      f'({time.time() - t0:.0f} s on {N_WORKERS} workers)')

## 6. Paper figures, tables and statistics

Each script in `paper_scripts/` reads `RUN_DIR` and writes to its own folder in `RESULTS_DIR`.

In [ ]:
ENV = {**os.environ, **{name: str(getattr(cfg, name)) for name in
       ('RAW_DIR', 'RUN_DIR', 'DISTANCE_DIR', 'RESULTS_DIR', 'SURROGATE_DIR', 'TEST_RETEST_DIR')}}


def run(*args):
    """Run one paper script with the notebook's paths; return its output."""
    t0 = time.time()
    r = subprocess.run([sys.executable, '-B', *args], cwd=REPO / 'paper_scripts', env=ENV,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f'{args} failed')
    print(f'[{" ".join(args)}: {time.time() - t0:.0f} s]')
    return r.stdout


def show(*parts):
    print((RESULTS_DIR.joinpath(*parts)).read_text())

### Figure 2 and the detection threshold (Section 2.2.1)

In [ ]:
run('cluster_verification.py')
show('figure2', 'template_correlations.txt')

### Table 1: cluster composition

In [ ]:
print(run('-m', 'participant_stats.table1_composition'))

### Table 4: retained recordings

In [ ]:
run('-m', 'participant_stats.build_recordings')
print(run('-m', 'participant_stats.table4_recordings'))

### Figure 3, Table 2 and the session statistics (Sections 2.2.2-2.2.3)

In [ ]:
out = run('main_analysis.py')
print(out[out.index('OVERLAP FILTERING SUMMARY'):out.index('ANALYSIS 1')])
print(out[out.index('SUBJECT-LEVEL PERMUTATION TESTS'):out.index('GENERATING FIGURE')])
print(run('-m', 'participant_stats.participant_counts'))
run('-m', 'participant_stats.participant_level')
import pandas as pd
print(pd.read_csv(RESULTS_DIR / 'participant_level' / 'models.csv').iloc[:2, :6].to_string(index=False))
print(pd.read_csv(RESULTS_DIR / 'participant_level' / 'paired_tests.csv').iloc[:1].to_string(index=False))

### Table 3: travelling/standing vs stable/oscillating

In [ ]:
print(run('travelling_polarity/travelling_polarity.py'))

### Figure 4: spectra around sequences (Section 2.2.4)

In [ ]:
run('spectral/compute_recording_alpha.py')
run('spectral/compute_sequence_spectra.py')
run('spectral/plot_sequence_spectra.py')
run('spectral/plot_alpha_vs_polarity.py')
show('spectral', 'sequence_spectra_stats.txt')
show('spectral', 'sequence_spectra_alpha_polarity_stats.txt')

### Figures 9 and 10: IAAFT surrogates (Section 2.1)

The surrogate repertoire is produced by a sequence of scripts run on a cluster
(`paper_scripts/surrogates/README.md`): `surrogates/pools.py` (one IAAFT surrogate per recording +
the detection of step 2), `surrogates/full_route.py blocks iaaft` (UOT + subDTW, ~100 CPU-hours),
`surrogates/full_route.py cluster iaaft` (the clustering of step 4). The scripts below read its
outputs from `SURROGATE_DIR`.

In [ ]:
print(run('surrogates/plot_iaaft_clusters.py'))
print(run('surrogates/plot_durations.py'))
print(run('surrogates/surrogate_numbers.py'))

### Figure 8: the test-retest dataset (Appendix A)

Wang et al. (2022), *A test-retest resting, and cognitive state EEG dataset during multiple
subject-driven states*, Scientific Data 9, 566 (OpenNeuro ds004148), eyes closed. Steps 1-4 of this
notebook were run on it as a sequence with the original code (`paper_scripts/test_retest/README.md`,
scripts in `paper_scripts/test_retest/original/`): preprocessing, detection, pool, UOT + subDTW,
clustering at the paper's percentile with minimum size 10. The two scripts below read that run from
`TEST_RETEST_DIR`.

In [ ]:
print(run('test_retest/percentile_cut.py'))
print(run('test_retest/plot_test_retest_clusters.py'))

## 7. Further figures (not in the paper)

In [ ]:
if MAKE_EXTRA_FIGURES:
    for cid in clusters_to_keep:
        stppy.visualization.plot_cluster_average(kept[cid], POS_FILE, cluster_id=cid, vlim=[-1, 1])
    stppy.visualization.plot_avalanche_topomaps(avalanches, POS_FILE, n_samples=10, random_seed=42)
    stppy.visualization.plot_sequence_topomaps(
        sequences_by_cluster, datasets, dataset_names, POS_FILE, cluster_id=clusters_to_keep[0],
        polarity='standing', sort_by_length=True, n_sequences=3)
    # Videos (require ffmpeg)
    stppy.visualization.create_cluster_video(
        kept[clusters_to_keep[0]], POS_FILE, output_file=str(RESULTS_DIR / 'cluster_0.mp4'),
        cluster_id=clusters_to_keep[0], fps=5)
    stppy.visualization.create_cluster_sequences_video(
        sequences_by_cluster, datasets, dataset_names, POS_FILE,
        output_file=str(RESULTS_DIR / 'cluster_0_sequences.mp4'), cluster_id=clusters_to_keep[0],
        polarity='standing', sort_by_length=True, n_sequences=10, fps=10)